# Debug `rotary_embedding_v2.py`

This notebook validates behavior of `hmr4d/network/base_arch/embeddings/rotary_embedding_v2.py` against the existing `rotary_embedding.py`, and runs ND RoPE sanity checks for 1D/2D/3D token layouts.

In [1]:
import torch

from hmr4d.network.base_arch.embeddings.rotary_embedding import (
    ROPE as ROPE_OLD,
    get_encoding as get_encoding_old,
)
from hmr4d.network.base_arch.embeddings.rotary_embedding_v2 import (
    ROPE as ROPE_NEW,
    get_encoding as get_encoding_new,
    get_1d_rotary_pos_embed,
    get_nd_rotary_pos_embed,
    apply_rotary_emb_qk,
)

torch.manual_seed(0)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)

/home/guangyu/patrick/GVHMR/hmr4d/network/base_arch/embeddings/rotary_embedding.py:14: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast(enabled=False)
/home/guangyu/patrick/GVHMR/hmr4d/network/base_arch/embeddings/rotary_embedding_v2.py:91: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  @autocast(enabled=False)


device: cuda


## 1) Backward compatibility checks (old vs new 1D API)

In [2]:
# Check get_encoding parity
for d_model, L in [(16, 32), (32, 128), (64, 256)]:
    e_old = get_encoding_old(d_model, L)
    e_new = get_encoding_new(d_model, L)
    max_diff = (e_old - e_new).abs().max().item()
    print(f'get_encoding d_model={d_model}, L={L}, max_diff={max_diff:.6g}')

get_encoding d_model=16, L=32, max_diff=0
get_encoding d_model=32, L=128, max_diff=0
get_encoding d_model=64, L=256, max_diff=0


In [3]:
# Check ROPE.rotate_queries_or_keys parity
tests = [(2, 8, 16, 64), (1, 4, 120, 32), (3, 2, 7, 16)]  # (B, H, L, D)
for B, H, L, D in tests:
    x = torch.randn(B, H, L, D, device=device)
    r_old = ROPE_OLD(D, max_seq_len=4096).to(device)
    r_new = ROPE_NEW(D, max_seq_len=4096).to(device)
    y_old = r_old.rotate_queries_or_keys(x)
    y_new = r_new.rotate_queries_or_keys(x)
    max_diff = (y_old - y_new).abs().max().item()
    print(f'ROPE parity (B,H,L,D)=({B},{H},{L},{D}) max_diff={max_diff:.6g}')

ROPE parity (B,H,L,D)=(2,8,16,64) max_diff=0
ROPE parity (B,H,L,D)=(1,4,120,32) max_diff=0
ROPE parity (B,H,L,D)=(3,2,7,16) max_diff=0


## 2) New API checks: 1D/2D/3D ND frequency generation

In [4]:
# 1D
freq_1d = get_1d_rotary_pos_embed(dim=64, pos=120, use_real=True)
print('1D cos/sin shapes:', freq_1d[0].shape, freq_1d[1].shape)

# 2D: rope_dim_list sums to head_dim
freq_2d = get_nd_rotary_pos_embed([16, 16], (8, 8), use_real=True)
print('2D cos/sin shapes:', freq_2d[0].shape, freq_2d[1].shape)  # [tokens, D]')

# 3D: e.g. T,H,W = 4,8,8; rope_dim_list sums to 32
freq_3d = get_nd_rotary_pos_embed([8, 12, 12], (4, 8, 8), use_real=True)
print('3D cos/sin shapes:', freq_3d[0].shape, freq_3d[1].shape)  # [tokens, D]')

1D cos/sin shapes: torch.Size([120, 64]) torch.Size([120, 64])
2D cos/sin shapes: torch.Size([64, 32]) torch.Size([64, 32])
3D cos/sin shapes: torch.Size([256, 32]) torch.Size([256, 32])


## 3) Apply ND rotary to Q/K tensors

In [ ]:
# Simulate flattened 3D tokens: S = T*H*W
B, S, Hh, D = 2, 4*8*8, 6, 32
xq = torch.randn(B, S, Hh, D, device=device)
xk = torch.randn(B, S, Hh, D, device=device)

cos, sin = get_nd_rotary_pos_embed([8, 12, 12], (4, 8, 8), use_real=True)
cos = cos.to(device)
sin = sin.to(device)

xq_rot, xk_rot = apply_rotary_emb_qk(xq, xk, (cos, sin), head_first=False)
print('xq_rot shape:', xq_rot.shape, 'xk_rot shape:', xk_rot.shape)
print('finite check:', torch.isfinite(xq_rot).all().item(), torch.isfinite(xk_rot).all().item())
print('mean |delta q|:', (xq_rot - xq).abs().mean().item())

## 4) Quick assertions for CI-like sanity

In [ ]:
# Hard assertions for parity and shape correctness
e_old = get_encoding_old(32, 128)
e_new = get_encoding_new(32, 128)
assert torch.allclose(e_old, e_new), 'get_encoding mismatch'

x = torch.randn(1, 4, 64, 32, device=device)
y_old = ROPE_OLD(32).to(device).rotate_queries_or_keys(x)
y_new = ROPE_NEW(32).to(device).rotate_queries_or_keys(x)
assert torch.allclose(y_old, y_new), 'ROPE.rotate_queries_or_keys mismatch'

cos3, sin3 = get_nd_rotary_pos_embed([8, 12, 12], (4, 8, 8), use_real=True)
assert cos3.shape == sin3.shape == (4*8*8, 32), 'ND RoPE shape mismatch'

print('All checks passed.')